# 公开数据获取与来源核验

本研究使用已经公开的去标识资料，不开展新增招募、动物或细胞实验。
这里记录数据来源、许可、校验和与实际下载状态。公开目录存在不等于图像质量合格，也不等于结局可靠。


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/rn_public').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from rn_public.bundle import restore_sources
print('Source restoration:', restore_sources(ROOT))
print('Project root:', ROOT.name)


Source restoration: {'restored_files': 0, 'status': 'all_source_checksums_verified'}
Project root: rn-mri-clinical-fusion


In [2]:
import hashlib
sources = json.loads((ROOT / 'data/metadata/source_files.json').read_text())
for item in sources:
    p = ROOT / item['local_path']
    actual = hashlib.sha256(p.read_bytes()).hexdigest()
    assert actual == item['sha256'], item['filename']
print(pd.DataFrame(sources)[['filename','size_bytes','license','status']].to_string(index=False))
print('All original OpenBTAI table/mask checksums match the acquisition manifest.')


                                filename  size_bytes license                       status
         OpenBTAI_METS_ClinicalData.xlsx      524828     CC0 downloaded_checksum_verified
OpenBTAI_MORPHOLOGICAL_MEASUREMENTS.xlsx      104053     CC0 downloaded_checksum_verified
                  OpenBTAI_Radiomic.xlsx    25055752     CC0 downloaded_checksum_verified
                        NIFTIs_Masks.zip    12394370     CC0 downloaded_checksum_verified
All original OpenBTAI table/mask checksums match the acquisition manifest.


In [3]:
archives = pd.read_json(ROOT / 'data/metadata/archive_sources.json')
print(archives[['archive_id','archive_name','archive_size','download_url']].to_string(index=False))
print('To restore small sources/directories: python scripts/download_sources.py')
print('To restore selected MRI: python scripts/download_selected_images.py --workers 6')


archive_id archive_name  archive_size                                    download_url
    dicom1 DICOMs_1.zip    4427620665 https://ndownloader.figshare.com/files/39603262
    dicom2 DICOMs_2.zip    7917887555 https://ndownloader.figshare.com/files/39603478
    dicom3 DICOMs_3.zip    6987829021 https://ndownloader.figshare.com/files/39603337
    dicom4 DICOMs_4.zip    5958611517 https://ndownloader.figshare.com/files/39604174
    dicom5 DICOMs_5.zip    7640221697 https://ndownloader.figshare.com/files/39604666
    dicom6 DICOMs_6.zip    5265464865 https://ndownloader.figshare.com/files/39604738
To restore small sources/directories: python scripts/download_sources.py
To restore selected MRI: python scripts/download_selected_images.py --workers 6


## 数据范围

OpenBTAI用于主要队列整理。UCSD、Yale和PROTEAS的公开临床文件仅作为可行性核验参考，不自动并入脑坏死预测队列。
源数据与派生数据分开保存。MRI体素保存在 `data/imaging/`，Excel保存数值、索引和审核状态。
